In [1]:
import Scope
from Scope.Read_Write import load_binary
from Scope.Classes_System import system
from Scope.Spin_Crossover.SCO_Classes import sco_system

In [2]:
from Scope.Other               import get_metal_idxs, get_metal_species
from Scope.Elementdata         import ElementData 
from Scope.Software.Quantum_Espresso.QE_Functions import get_QE_data, get_pp
elemdatabase = ElementData()

In [3]:
import Scope.Software.Quantum_Espresso 

In [4]:
test_input = """
&environment
   max_jobs         = 400
   max_procs        = 3200
   requested_procs  = 32
/

&options
   want_submit       = True
   ignore_submitted  = False
   overwrite_inputs  = False
   overwrite_logs    = False
/

&job_data
   branch       = 'Solid'
   target       = 'ref_crys'
   setup        = 'regular'
   suffix       = 'relax'
   keyword      = 'relax'

   istate       = 'initial'
   fstate       = 'relax'

   hierarchy    = 1
   requisites   = []
   constrains   = ['self']
   must_be_good = True
/

&qc_data
   software     = 'qe'
   version      = '7.0'
   jobtype      = 'relax'
   functional   = 'pbe'
   is_hubbard   = True
   is_grimme    = True
   uterm        = 2.35
   cutoff       = 25
   mix_beta     = 0.7
   pseudo       = '/home/svela/Programes/PP_Library'
   ## If "spin" is not specified, a default will be taken
   ## Maybe also PP_Library
/
"""

In [5]:
from Scope.Classes_Input import *
local_env   = set_environment_data(content=test_input, section="&environment", isfile=False)
options     = set_options_data(content=test_input, section="&options", isfile=False)
job_data    = set_job_data(content=test_input, section="&job_data", isfile=False)
qc_data     = set_qc_data(content=test_input, section="&qc_data", isfile=False)

In [6]:
sys = load_binary("/Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/ABITEM.npy")

In [7]:
sys

-------------------------------------
-- >>> SCOPE SCO-System Object >>> --
-------------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = sco_system
 Name                  = ABITEM
 Source Path           = /Users/sergivela/Documents/SCOPE/Database_SCO/4-Merged/ABITEM/
 Calculations Path     = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/
 System File Path      = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/
 System File Name      = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/ABITEM.npy

 # of Sources          = 6
     idx, type, name, formula               
     0: cell ABITEM01 H104-C92-N24-O4-S8-Fe4 
     1: cell ABITEM H104-C92-N24-O4-S8-Fe4 
     2: cell ref_hs_cell H104-C92-N24-O4-S8-Fe4 
     3: cell ref_ls_cell H104-C92-N24-O4-S8-Fe4 
     4: specie ref_hs_mol H18-C20-N6-S2-Fe 
     5: specie ref_ls_mol H18-C20-N6-S2-Fe 


In [8]:
sou = sys.sources[4]
print(sou)

--------------------------------------------------
------------- SCOPE MOLECULE Object --------------
--------------------------------------------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule
 Number of Atoms       = 47
 Formula               = H18-C20-N6-S2-Fe
 Charge                = 0
 Spin                  = 0
 SMILES                = ['[N-]=C=S', '[H]c1nc(C([H])([H])N(C([H])([H])c2nc([H])c([H])c([H])c2[H])C([H])([H])c2nc([H])c([H])c([H])c2[H])c([H])c([H])c1[H]', '[N-]=C=S']
 Number of Parents     = 1
 Has Adjacency Matrix  = YES
 Has Bonds             = YES
 # Ligands             = 3
 # Metals              = 1




In [9]:
this_branch = sys.add_branch("Isolated")
rec_hs = this_branch.add_recipe("ref_hs_mol")
rec_ls = this_branch.add_recipe("ref_ls_mol")
job_hs = rec_hs.add_job(job_data)
job_ls = rec_ls.add_job(job_data)

In [10]:
exists, sou_hs = sys.find_source("ref_hs_mol")
exists, sou_ls = sys.find_source("ref_ls_mol")
sou_hs.set_spin_config(4, typ='metal', debug=1)
sou_ls.set_spin_config(0, typ='metal', debug=1)

SPECIE.SET_SPIN_CONFIG: Preparing Spin Configuration for Specie H18-C20-N6-S2-Fe
SPECIE.SET_SPIN_CONFIG: Received spins 4
SPECIE.SET_SPIN_CONFIG: Received typ metal
SPECIE.SET_SPIN_CONFIG: Setting spin=4 to metal atom Fe
SPECIE.SET_SPIN_CONFIG: Preparing Spin Configuration for Specie H18-C20-N6-S2-Fe
SPECIE.SET_SPIN_CONFIG: Received spins 0
SPECIE.SET_SPIN_CONFIG: Received typ metal
SPECIE.SET_SPIN_CONFIG: Setting spin=0 to metal atom Fe


0

In [11]:
exists, istate = sou_hs.find_state(job_hs.job_data.istate)

In [12]:
istate._source

--------------------------------------------------
------------- SCOPE MOLECULE Object --------------
--------------------------------------------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule
 Number of Atoms       = 47
 Formula               = H18-C20-N6-S2-Fe
 Charge                = 0
 Spin                  = 4
 SMILES                = ['[N-]=C=S', '[H]c1nc(C([H])([H])N(C([H])([H])c2nc([H])c([H])c([H])c2[H])C([H])([H])c2nc([H])c([H])c([H])c2[H])c([H])c([H])c1[H]', '[N-]=C=S']
 Number of Parents     = 1
 Has Adjacency Matrix  = YES
 Has Bonds             = YES
 # Ligands             = 3
 # Metals              = 1


In [13]:
for at in istate._source.atoms:
    if at.spin > 0: print(at.label, at.spin)

Fe 4


In [14]:
istate.get_atoms()
for at in istate.atoms:
    if at.spin > 0: print(at.label, at.spin)

Fe 4


In [15]:
exists, this_branch = sys.find_branch("solid")
print(this_branch)

---------------------------------------------------
   >>> BRANCH                                      
---------------------------------------------------
 System                = ABITEM
---------------------------------------------------
 self.status           = active
 self.creation_time    = 21/10/2025 12:17:41
 self.creation_user    = sergivela
 self.path             = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/
 self.name             = Solid
 Num Recipes           = 2




In [16]:
this_branch.recipes

[---------------------------------------------------
    >>> >>> RECIPE                                  
 ---------------------------------------------------
  Source Name                 = ref_hs_cell
  Source Spin                 = None
  Source Phase                = HS
  Source Type                 = cell
  Source sub-Type             = sco_cell
 ---------------------------------------------------
  Recipe Name (from Source)   = ref_hs_cell
  Num Jobs                    = 1
 	Last Job Keyword     = relax
 	Last Job Hierarchy   = 1
 ,
 ---------------------------------------------------
    >>> >>> RECIPE                                  
 ---------------------------------------------------
  Source Name                 = ref_ls_cell
  Source Spin                 = None
  Source Phase                = LS
  Source Type                 = cell
  Source sub-Type             = sco_cell
 ---------------------------------------------------
  Recipe Name (from Source)   = ref_ls_cell
  Num

In [17]:
exists, this_recipe = this_branch.find_recipe("ref_hs_cell")
print(this_recipe)

---------------------------------------------------
   >>> >>> RECIPE                                  
---------------------------------------------------
 Source Name                 = ref_hs_cell
 Source Spin                 = None
 Source Phase                = HS
 Source Type                 = cell
 Source sub-Type             = sco_cell
---------------------------------------------------
 Recipe Name (from Source)   = ref_hs_cell
 Num Jobs                    = 1
	Last Job Keyword     = relax
	Last Job Hierarchy   = 1




In [18]:
exists, this_job = this_recipe.find_job("relax")
print(this_job)

---------------------------------------------------
   >>> >>> >>> JOB                                 
---------------------------------------------------
 Source Type           = cell
 Source Spin           = None
 Branch Name           = Solid
 Recipe Name           = ref_hs_cell
---------------------------------------------------
 Job path              = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/
 Job keyword           = relax
 Job hierarchy         = 1
 Job requisites        = []
 Job constrains        = ['self']
 Job setup             = regular
 Num Computations      = 1
----------------------------------------------------
 self.isregistered (Temp) = True
 self.isgood       (Temp) = False
 self.isfinished   (Temp) = False




In [19]:
exists, this_comp = this_job.find_computation(run_number=1)
print(this_comp)

---------------------------------------------------
   >>> >>> >>> >>> COMPUTATION                     
---------------------------------------------------
 Source Type           = cell
 Source sub-Type       = sco_cell
 Branch Name           = Solid
 Recipe Name           = ref_hs_cell
 Job Keyword           = relax
---------------------------------------------------
 Initial State         = initial
 Final State           = relax
 Comp software         = qe
 Comp index            = 1
 Comp step             = 1
 Comp run_number       = 1
 Comp keyword          = 
 Comp inp_path         = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/ABITEM_ref_hs_cell_relax_r1.input
 Comp out_path         = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Solid/ABITEM_ref_hs_cell_relax_r1.out
 Comp isregistered     = False




In [20]:
this_comp.source.charge

In [21]:
exists, istate = this_comp.source.find_state(this_comp.qc_data.istate)
print(istate)

---------------------------------------------------
   STATE                                           
---------------------------------------------------
 Name                  = initial
 Source Name           = ref_hs_cell
 Source Type           = cell
 Labels                = Fe...
 Coord                 = [3.7430176, 3.9944543, 13.5094621]...
 Number of Complexes   = 4
 # Molecules:          = 8
 With Formulae:                               
    0: H8-C3-O 
    1: H8-C3-O 
    2: H8-C3-O 
    3: H8-C3-O 
    4: H18-C20-N6-S2-Fe 
    5: H18-C20-N6-S2-Fe 
    6: H18-C20-N6-S2-Fe 
    7: H18-C20-N6-S2-Fe 



In [22]:
istate._source

-----------------------------------
-- >>> SCOPE SCO-CELL Object >>> --
-----------------------------------
 Version               = 1.0
 Type                  = cell
 SubType               = sco_cell
 Name                  = ref_hs_cell
 Num Atoms             = 236
 Cell Parameters a:c   = [12.3008, 11.9241, 17.6916]
 Cell Parameters al:ga = [90.0, 92.704, 90.0]
 # Molecules:          = 8
 With Formulae:                               
    0: H18-C20-N6-S2-Fe 
    1: H18-C20-N6-S2-Fe 
    2: H8-C3-O 
    3: H8-C3-O 
    4: H8-C3-O 
    5: H8-C3-O 
    6: H18-C20-N6-S2-Fe 
    7: H18-C20-N6-S2-Fe 
-------------------------------
 # of Ref Molecules:   = 2
 With Formulae:                                  
    0: H18-C20-N6-S2-Fe 
    1: H8-C3-O 
 Phase                 = HS
 HS Molar Fraction     = 1.0
